# 05 - Feature Engineering (Phase 2)

Closes the backlog items from Feature-Engineering-Playbook.md by
actually measuring each one's effect, using XGBoost (fastest + best
model from Experiments 002-004) as the test harness, on the exact same
5-fold split used throughout this project. Each step is additive and
compared against the previous best, not against the raw baseline every
time - the question at each step is "does this specific idea help on
top of what already works," not "is this idea good in isolation.

In [ ]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

train = pd.read_csv("../Data/train.csv")
test = pd.read_csv("../Data/test.csv")
TARGET = "Will_Buy_EV"
y = train[TARGET].map({"No": 0, "Yes": 1})
base_cat_cols = ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level"]

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
folds = list(skf.split(train, y))
print(len(folds), "folds -", train.shape, test.shape)

5 folds - (668665, 15) (286571, 14)


## Reusable CV harness

Same XGBoost config as Experiment 004 (`max_depth=6`,
`learning_rate=0.05`, GPU, early stopping) wrapped in a function so each
feature set can be scored with one call, using the *same* fold indices
every time.

In [ ]:
def run_xgb_cv(X, Xt, tag=""):
    oof_pred = np.zeros(len(X))
    test_preds = np.zeros((N_SPLITS, len(Xt)))
    fold_aucs = []
    t0 = time.time()
    for fold, (tr_idx, va_idx) in enumerate(folds):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        model = XGBClassifier(
            n_estimators=3000, learning_rate=0.05, max_depth=6,
            tree_method="hist", device="cuda", enable_categorical=True,
            eval_metric="auc", early_stopping_rounds=50, random_state=42,
        )
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        va_pred = model.predict_proba(X_va)[:, 1]
        oof_pred[va_idx] = va_pred
        fold_aucs.append(roc_auc_score(y_va, va_pred))
        test_preds[fold] = model.predict_proba(Xt)[:, 1]
    oof_auc = roc_auc_score(y, oof_pred)
    print(f"[{tag}] mean_fold_auc={np.mean(fold_aucs):.6f} std={np.std(fold_aucs):.6f} OOF={oof_auc:.6f} time={time.time()-t0:.0f}s")
    return oof_auc, fold_aucs, oof_pred, test_preds.mean(axis=0)

X0 = train.drop(columns=[TARGET, "id"]).copy()
Xt0 = test.drop(columns=["id"]).copy()
for c in base_cat_cols:
    X0[c] = X0[c].astype("category")
    Xt0[c] = pd.Categorical(Xt0[c], categories=X0[c].cat.categories)

res_baseline = run_xgb_cv(X0, Xt0, "baseline (raw)")

[baseline (raw)] mean_fold_auc=0.941728 std=0.000794 OOF=0.941720 time=13s


Reproduces Experiment 004 exactly (0.941720) - confirms the harness is
correct before trusting deltas measured with it.

## Step 1: income digit decomposition

`Annual_Income_USD` is always an integer-valued float (`% 1 == 0` for
100% of rows), range 30,000-188,549. Decompose into one categorical
feature per decimal digit position - a known Playground Series trick
for exploiting artifacts in how the synthetic data generator assigned
these values.

In [ ]:
def add_income_digits(df):
    v = df["Annual_Income_USD"].astype(int)
    return pd.DataFrame({f"inc_d{i}": ((v // (10**i)) % 10).astype("category") for i in range(6)}, index=df.index)

digits_tr = add_income_digits(train)
digits_te = add_income_digits(test)

X1 = pd.concat([X0, digits_tr], axis=1)
Xt1 = pd.concat([Xt0, digits_te], axis=1)

res_income_digits = run_xgb_cv(X1, Xt1, "income digits")

[income digits] mean_fold_auc=0.943959 std=0.000631 OOF=0.943955 time=20s


**+0.002235 AUC - the single biggest lever found in this project so
far.** Quantifies the backlog item precisely: not "digit decomposition
probably helps" but +0.0022, measured on identical folds.

## Step 2: commute digit decomposition (tested, dropped)

`Daily_Commute_km` has one decimal place (range 5.0-98.7). Same
decomposition idea, on top of the income digits (not replacing them).

In [ ]:
def add_commute_digits(df):
    v = (df["Daily_Commute_km"] * 10).round().astype(int)
    return pd.DataFrame({
        "com_tenths": (v % 10).astype("category"),
        "com_ones": ((v // 10) % 10).astype("category"),
        "com_tens": ((v // 100) % 10).astype("category"),
    }, index=df.index)

cdigits_tr = add_commute_digits(train)
cdigits_te = add_commute_digits(test)

X2 = pd.concat([X1, cdigits_tr], axis=1)
Xt2 = pd.concat([Xt1, cdigits_te], axis=1)

res_both_digits = run_xgb_cv(X2, Xt2, "income+commute digits")

[income+commute digits] mean_fold_auc=0.943913 std=0.000662 OOF=0.943910 time=22s


**-0.000045 vs. income-digits-only: no help, within noise. Dropped.**
This closes the other half of the backlog item honestly - the digit
trick worked for income specifically, not as a generic "decompose any
numeric column" move. Likely explanation: income's generation process
had digit-level artifacts; commute's didn't, or the 1-decimal-place
representation doesn't carry the same kind of signal. Commute digits are
not included in any feature set from here on.

## Step 3: frequency encoding (income + commute)

Count each raw value's frequency in the *training* data - high spread
in these counts (some exact income/commute values repeat far more than
others) can itself be a synthetic-generation artifact. Built on top of
income digits (Step 1), not commute digits (dropped in Step 2).

In [ ]:
def add_freq_features(df_tr, df_te, col):
    freq = df_tr[col].value_counts()
    return df_tr[col].map(freq).rename(f"{col}_freq"), df_te[col].map(freq).fillna(0).rename(f"{col}_freq")

inc_freq_tr, inc_freq_te = add_freq_features(train, test, "Annual_Income_USD")
com_freq_tr, com_freq_te = add_freq_features(train, test, "Daily_Commute_km")

X3 = pd.concat([X1, inc_freq_tr, com_freq_tr], axis=1)
Xt3 = pd.concat([Xt1, inc_freq_te, com_freq_te], axis=1)

res_freq = run_xgb_cv(X3, Xt3, "income digits + freq enc")

[income digits + freq enc] mean_fold_auc=0.944182 std=0.000555 OOF=0.944174 time=24s


**+0.000219 vs. income-digits-only: real, kept. Current best: 0.944174
OOF** (+0.002454 over the raw Experiment 004 baseline of 0.941720).

## Step 4: in-fold target encoding (tested, dropped)

Mean target per category, refit inside each fold's training split only
(no leakage across the fold boundary) - the technique documented in
Model-Zoo-and-CV-Strategy.md, applied here to the 6 base categorical
columns on top of the Step 3 feature set.

In [ ]:
def run_xgb_cv_with_te(X, Xt, te_cols, tag=""):
    oof_pred = np.zeros(len(X))
    test_preds = np.zeros((N_SPLITS, len(Xt)))
    fold_aucs = []
    t0 = time.time()
    global_mean = y.mean()
    for fold, (tr_idx, va_idx) in enumerate(folds):
        X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
        X_te_test = Xt.copy()
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        for col in te_cols:
            means = y_tr.groupby(X_tr[col], observed=True).mean()
            # NOTE: .map() on a categorical Series returns a categorical
            # RESULT (with float categories!) in this pandas version -
            # must force back to plain float, or XGBoost's
            # enable_categorical auto-detection treats it as a broken
            # categorical column and raises "Category index ... has
            # floating point dtype".
            X_tr[f"{col}_te"] = X_tr[col].map(means).astype(float).fillna(global_mean)
            X_va[f"{col}_te"] = X_va[col].map(means).astype(float).fillna(global_mean)
            X_te_test[f"{col}_te"] = X_te_test[col].map(means).astype(float).fillna(global_mean)
        model = XGBClassifier(
            n_estimators=3000, learning_rate=0.05, max_depth=6,
            tree_method="hist", device="cuda", enable_categorical=True,
            eval_metric="auc", early_stopping_rounds=50, random_state=42,
        )
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        va_pred = model.predict_proba(X_va)[:, 1]
        oof_pred[va_idx] = va_pred
        fold_aucs.append(roc_auc_score(y_va, va_pred))
        test_preds[fold] = model.predict_proba(X_te_test)[:, 1]
    oof_auc = roc_auc_score(y, oof_pred)
    print(f"[{tag}] mean_fold_auc={np.mean(fold_aucs):.6f} std={np.std(fold_aucs):.6f} OOF={oof_auc:.6f} time={time.time()-t0:.0f}s")
    return oof_auc, fold_aucs, oof_pred, test_preds.mean(axis=0)

res_te = run_xgb_cv_with_te(X3, Xt3, base_cat_cols, "income digits + freq + target enc")

[income digits + freq + target enc] mean_fold_auc=0.944147 std=0.000553 OOF=0.944137 time=26s


**-0.000037 vs. Step 3: no help, dropped.** Makes sense in hindsight:
XGBoost's native categorical splits (`enable_categorical=True`) can
already extract most of what a mean-target column would add for these
6 low-cardinality categoricals - the extra column is redundant signal,
not new signal.

## Step 5: City_Type deviation feature (tested, dropped)

From the Simpson's-paradox finding in EDA-Findings.md:
`Charging_Stations_Near_Home/Work` correlate with each other partly
*because* both depend on `City_Type`. Tested an explicit feature - each
row's charging-station count minus its `City_Type` group's mean - to
see if making that dependency explicit helps beyond what the tree
already extracts from `City_Type` + the raw counts.

In [ ]:
def add_city_deviation(df_tr, df_te):
    home_mean = df_tr.groupby("City_Type")["Charging_Stations_Near_Home"].transform("mean")
    work_mean = df_tr.groupby("City_Type")["Charging_Stations_Near_Work"].transform("mean")
    home_map = df_tr.groupby("City_Type")["Charging_Stations_Near_Home"].mean()
    work_map = df_tr.groupby("City_Type")["Charging_Stations_Near_Work"].mean()
    tr = pd.concat([
        (df_tr["Charging_Stations_Near_Home"] - home_mean).rename("home_dev_from_city"),
        (df_tr["Charging_Stations_Near_Work"] - work_mean).rename("work_dev_from_city"),
    ], axis=1)
    te = pd.concat([
        (df_te["Charging_Stations_Near_Home"] - df_te["City_Type"].map(home_map)).rename("home_dev_from_city"),
        (df_te["Charging_Stations_Near_Work"] - df_te["City_Type"].map(work_map)).rename("work_dev_from_city"),
    ], axis=1)
    return tr, te

dev_tr, dev_te = add_city_deviation(train, test)
X4 = pd.concat([X3, dev_tr], axis=1)
Xt4 = pd.concat([Xt3, dev_te], axis=1)

res_citydev = run_xgb_cv(X4, Xt4, "income digits + freq + city-deviation")

[income digits + freq + city-deviation] mean_fold_auc=0.944102 std=0.000626 OOF=0.944097 time=19s


**-0.000077 vs. Step 3: no help, dropped.** Same story as target
encoding: `City_Type` is already a feature the tree splits on directly,
so an explicit deviation column doesn't add information the model
couldn't already reach on its own. Closes the Simpson's-paradox backlog
item - the paradox was a useful thing to *understand* about the data,
but didn't translate into a useful engineered feature here.

## Summary

| Step | Change | OOF AUC | Δ vs. base of comparison | Kept? |
|---|---|---|---|---|
| 0 | Baseline (raw, Exp. 004) | 0.941720 | - | - |
| 1 | + income digit decomposition | 0.943955 | +0.002235 vs. Step 0 | **Yes** |
| 2 | + commute digit decomposition | 0.943910 | -0.000045 vs. Step 1 | No |
| 3 | + frequency encoding (income+commute) | **0.944174** | +0.000219 vs. Step 1 | **Yes** |
| 4 | + target encoding (6 categoricals) | 0.944137 | -0.000037 vs. Step 3 | No |
| 5 | + City_Type deviation features | 0.944097 | -0.000077 vs. Step 3 | No |

**Winning feature set: income digit decomposition + frequency encoding
of income and commute. OOF AUC 0.944174**, a **+0.002454** improvement
over the Experiment 004 raw-feature baseline (0.941720) - by far the
largest single gain in this project so far, bigger than the combined
effect of trying three different model families (which spanned less
than 0.0003 AUC end to end).

In [ ]:
test_pred_best = res_freq[3]

submission = pd.DataFrame({"id": test["id"], TARGET: test_pred_best})
submission.to_csv("../submissions/xgboost_5fold_income_digits_freq.csv", index=False)

print("Submission shape:", submission.shape)
print("Missing values:", submission.isna().sum().sum())
submission.head()

       id  Will_Buy_EV\n0  668665     0.017888\n1  668666     0.011543\n2  668667     0.006681\n3  668668     0.004141\n4  668669     0.021297

## Experiment 005 - Feature Engineering on XGBoost

- Base model: `XGBClassifier`, same config and same 5-fold split as
  Experiment 004
- Winning features: income digit decomposition (6 columns) + frequency
  encoding of raw income and commute values (2 columns), on top of the
  13 raw features
- Tested and explicitly dropped: commute digit decomposition, in-fold
  target encoding of the 6 base categoricals, `City_Type` deviation
  features - each measured, not guessed, and each made things very
  slightly worse or made no real difference

**OOF AUC: 0.944174** (vs. 0.941720 raw XGBoost baseline, Experiment 004)

**Next:** submit `xgboost_5fold_income_digits_freq.csv` to confirm the
OOF gain shows up on the public LB (expected, given the CV/LB
relationship has held exactly on 3/3 submissions so far). After that,
Phase 3's Optuna tuning and Phase 4's calibration/stacking both now have
a meaningfully stronger feature set to build on - stacking in particular
should revisit whether CatBoost/LightGBM on this same feature set
produce more diverse predictions than they did on raw features.